In [ ]:
import subprocesssubprocess.run(['pip', 'install', 'xgboost', 'lightgbm', 'shap', 'lime',                'imbalanced-learn', 'scikit-learn', '-q'], check=False)import os, gc, warnings, joblib, jsonimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesfrom pathlib import Pathfrom collections import Counterfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import RobustScaler, LabelEncoderfrom sklearn.feature_selection import SelectKBest, f_classiffrom sklearn.ensemble import RandomForestClassifier, VotingClassifierfrom sklearn.metrics import (    accuracy_score, f1_score, precision_score, recall_score,    classification_report, confusion_matrix, ConfusionMatrixDisplay)import xgboost as xgbimport lightgbm as lgbimport shapimport limeimport lime.lime_tabularfrom imblearn.over_sampling import SMOTEwarnings.filterwarnings('ignore')pd.set_option('display.max_columns', 50)plt.rcParams['figure.dpi'] = 120OUT = Path('/kaggle/working/xai_ids_outputs')for sub in ['plots', 'models', 'explanations']:    (OUT / sub).mkdir(parents=True, exist_ok=True)print('All packages imported')try:    import torch    g = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'    print('GPU:', g)except Exception:    print('torch not available')

In [ ]:
USE_CICIDS2017 = TrueUSE_UNSWNB15   = TrueUSE_CICIDS2018 = TrueMAX_PER_DATASET = {    'CICIDS2017': 200_000,    'UNSWNB15':   145_000,    'CICIDS2018': 200_000,}CICIDS2018_ROWS_PER_FILE = 20_000TOP_K_FEATURES = 20TEST_SIZE      = 0.20RANDOM_STATE   = 42XGB_EPOCHS     = 300PATIENCE       = 20XCS_W1        = 0.4XCS_W2        = 0.3XCS_W3        = 0.3XCS_THRESHOLD = 0.3N_XCS_SAMPLES = 86def find_path(candidates):    for c in candidates:        p = Path(c)        if p.exists():            return p    return NonePATHS = {    'cicids2017': find_path([        '/kaggle/input/datasets/mohdqwe123/dataset1/cicids2017',        '/kaggle/input/datasets/dhoogla/cicids2017',        '/kaggle/input/cicids2017',        '/kaggle/input/network-intrusion-dataset',    ]),    'unswnb15': find_path([        '/kaggle/input/datasets/mohdqwe123/dataset1/unswnb15',        '/kaggle/input/datasets/dhoogla/unswnb15',        '/kaggle/input/unswnb15',        '/kaggle/input/unsw-nb15',    ]),    'cicids2018': find_path([        '/kaggle/input/datasets/mohdqwe123/dataset1/csecicids2018',        '/kaggle/input/datasets/solarmainframe/ids-intrusion-csv',        '/kaggle/input/ids-intrusion-csv',        '/kaggle/input/csecicids2018',    ]),}print('Dataset paths:')for k, v in PATHS.items():    print(' ', k, ':', v if v else 'NOT FOUND')print('\nAll /kaggle/input top-level dirs:')for p in sorted(Path('/kaggle/input').iterdir()):    if p.is_dir():        print(' ', p.name)

In [ ]:
ALWAYS_DROP = [    'Flow ID', 'Source IP', 'Destination IP', 'Src IP', 'Dst IP',    'Source Port', 'Destination Port', 'Timestamp', 'timestamp',    'proto', 'service', 'state', 'id', 'srcip', 'dstip',    'label', 'label_val', 'attack', 'attack_cat', 'is_attack', 'target',]LABEL_CANDIDATES = [    'Label', 'label', ' Label', 'attack_cat', 'Attack', 'attack', 'type', 'label_val']def clean_dataset(raw, max_samples=None):    print('  raw shape:', raw.shape)    raw = raw.copy()    raw.replace([np.inf, -np.inf], np.nan, inplace=True)    raw.dropna(inplace=True)    raw.drop_duplicates(inplace=True)    drop_cols = [c for c in ALWAYS_DROP if c in raw.columns]    X = raw.drop(columns=['Label'] + drop_cols, errors='ignore')    for col in X.select_dtypes(include=['object']).columns:        X[col] = pd.to_numeric(X[col], errors='coerce')    X = X.select_dtypes(include=[np.number])    zero_var = X.columns[X.std() == 0].tolist()    if zero_var:        X = X.drop(columns=zero_var)        print('  dropped', len(zero_var), 'zero-variance cols')    X.replace([np.inf, -np.inf], np.nan, inplace=True)    mask  = X.notna().all(axis=1)    X     = X[mask].reset_index(drop=True)    y_raw = raw['Label'].astype(str).str.strip()[mask].reset_index(drop=True)    if max_samples and len(X) > max_samples:        rng = np.random.RandomState(RANDOM_STATE)        idx = rng.choice(len(X), max_samples, replace=False)        X     = X.iloc[idx].reset_index(drop=True)        y_raw = y_raw.iloc[idx].reset_index(drop=True)    le    = LabelEncoder()    y_enc = le.fit_transform(y_raw)    print('  clean shape:', X.shape, '| classes:', list(le.classes_))    return X, y_enc, ledef load_parquet_files(directory, max_total):    files = sorted(directory.rglob('*.parquet'))    print(' ', len(files), 'parquet files')    if not files:        return None    n_per = max_total // len(files)    dfs   = []    for f in files:        df = pd.read_parquet(f)        df.columns = df.columns.str.strip()        if len(df) > n_per:            df = df.sample(n=n_per, random_state=RANDOM_STATE)        dfs.append(df)        print('   sampled', f.name, ':', len(df))    return pd.concat(dfs, ignore_index=True)def load_csv_files_stratified(directory, rows_per_file):    files = sorted(directory.rglob('*.csv'))    print(' ', len(files), 'CSV files')    if not files:        return None    all_dfs = []    for f in files:        try:            enc = 'utf-8'            try:                head = pd.read_csv(f, nrows=5, low_memory=False, encoding=enc)            except UnicodeDecodeError:                enc  = 'latin-1'                head = pd.read_csv(f, nrows=5, low_memory=False, encoding=enc)            head.columns = head.columns.str.strip()            label_col = next((c for c in LABEL_CANDIDATES if c in head.columns), None)            if label_col is None:                print('   skip', f.name, '(no label)')                continue            class_buckets = {}            for chunk in pd.read_csv(f, encoding=enc, low_memory=False, chunksize=100_000):                chunk.columns = chunk.columns.str.strip()                chunk.replace([np.inf, -np.inf], np.nan, inplace=True)                chunk.dropna(inplace=True)                if label_col not in chunk.columns:                    continue                for lbl, grp in chunk.groupby(label_col):                    key = str(lbl).strip()                    if key not in class_buckets:                        class_buckets[key] = []                    class_buckets[key].append(grp)            if not class_buckets:                continue            n_cls  = len(class_buckets)            per_cls = max(50, rows_per_file // n_cls)            sampled = []            for key, parts in class_buckets.items():                full = pd.concat(parts, ignore_index=True)                n_tk = min(per_cls, len(full))                sampled.append(full.sample(n=n_tk, random_state=RANDOM_STATE))            file_df = pd.concat(sampled, ignore_index=True)            all_dfs.append(file_df)            print('   sampled', f.name, ':', len(file_df),                  '| labels:', list(class_buckets.keys()))        except Exception as e:            print('   ERROR', f.name, ':', e)    if not all_dfs:        return None    return pd.concat(all_dfs, ignore_index=True)def load_and_clean(directory, name, max_total, rows_per_csv, label_candidates=None):    directory     = Path(directory)    pq_files      = list(directory.rglob('*.parquet'))    csv_files     = list(directory.rglob('*.csv'))    if pq_files:        raw = load_parquet_files(directory, max_total)    elif csv_files:        raw = load_csv_files_stratified(directory, rows_per_csv)    else:        print('  No files in', directory)        return None, None, None    if raw is None or len(raw) == 0:        return None, None, None    cands     = label_candidates if label_candidates else LABEL_CANDIDATES    label_col = next((c for c in cands if c in raw.columns), None)    if label_col is None:        print('  No label column. Cols:', list(raw.columns)[:10])        return None, None, None    raw = raw.rename(columns={label_col: 'Label'})    print('  label col:', label_col, '| rows:', len(raw))    X, y, le = clean_dataset(raw, max_total)    del raw; gc.collect()    return X, y, leDATASETS = {}if USE_CICIDS2017 and PATHS['cicids2017']:    print('\nLoading CIC-IDS-2017...')    X, y, le = load_and_clean(        PATHS['cicids2017'], 'CICIDS2017',        max_total=MAX_PER_DATASET['CICIDS2017'], rows_per_csv=25_000)    if X is not None:        DATASETS['CICIDS2017'] = {'X': X, 'y': y, 'le': le}        print('  MB:', round(X.memory_usage(deep=True).sum() / 1e6))if USE_UNSWNB15 and PATHS['unswnb15']:    print('\nLoading UNSW-NB15...')    X, y, le = load_and_clean(        PATHS['unswnb15'], 'UNSWNB15',        max_total=MAX_PER_DATASET['UNSWNB15'], rows_per_csv=50_000,        label_candidates=['attack_cat', 'Label', 'label', 'label_val', 'Attack'])    if X is not None:        DATASETS['UNSWNB15'] = {'X': X, 'y': y, 'le': le}        print('  MB:', round(X.memory_usage(deep=True).sum() / 1e6))if USE_CICIDS2018 and PATHS['cicids2018']:    print('\nLoading CSE-CIC-IDS-2018 (stratified per-label per-file)...')    X, y, le = load_and_clean(        PATHS['cicids2018'], 'CICIDS2018',        max_total=MAX_PER_DATASET['CICIDS2018'],        rows_per_csv=CICIDS2018_ROWS_PER_FILE)    if X is not None:        DATASETS['CICIDS2018'] = {'X': X, 'y': y, 'le': le}        print('  MB:', round(X.memory_usage(deep=True).sum() / 1e6))print('\nDatasets loaded:', list(DATASETS.keys()))if not DATASETS:    raise RuntimeError('No datasets loaded — check PATHS in Cell 2')

In [ ]:
def select_features(X, y, k=20, name='ds'):    n_avail = X.shape[1]    k_use   = min(k, n_avail)    k_anova = min(30, n_avail)    print('  feature selection:', name, n_avail, '->', k_use)    if n_avail <= k_use:        imp = pd.Series(np.ones(n_avail), index=X.columns)        return X.columns.tolist(), imp    rng = np.random.RandomState(RANDOM_STATE)    n_s = min(100_000, len(X))    idx = rng.choice(len(X), n_s, replace=False)    sel = SelectKBest(f_classif, k=k_anova)    sel.fit(X.iloc[idx], y[idx])    top_cols = X.columns[sel.get_support()].tolist()    Xs    = X[top_cols].iloc[idx]    ys    = y[idx]    n_cls = len(np.unique(ys))    obj   = 'binary:logistic' if n_cls == 2 else 'multi:softprob'    qm = xgb.XGBClassifier(        n_estimators=80, max_depth=5, n_jobs=-1,        objective=obj, eval_metric='mlogloss',        random_state=RANDOM_STATE, verbosity=0    )    split = int(n_s * 0.8)    qm.fit(Xs.iloc[:split], ys[:split])    exp = shap.TreeExplainer(qm)    sv  = exp.shap_values(Xs.iloc[:1500])    if isinstance(sv, list):        ms = np.mean([np.abs(s).mean(0) for s in sv], axis=0)    elif sv.ndim == 3:        ms = np.abs(sv).mean(axis=(0, 2))    else:        ms = np.abs(sv).mean(0)    imp   = pd.Series(ms, index=top_cols).sort_values(ascending=False)    feats = imp.head(k_use).index.tolist()    print('  selected', len(feats), 'features')    return feats, impfor name, ds in DATASETS.items():    print('\nFeature selection:', name)    feats, imp = select_features(ds['X'], ds['y'], k=TOP_K_FEATURES, name=name)    ds['features']        = feats    ds['shap_importance'] = imp    ds['class_names']     = list(ds['le'].classes_)    plt.figure(figsize=(9, 6))    imp.head(TOP_K_FEATURES).sort_values().plot(kind='barh', color='#4C72B0')    plt.title(name + ' — Top ' + str(TOP_K_FEATURES) + ' Features (SHAP)')    plt.xlabel('Mean |SHAP value|')    plt.tight_layout()    plt.savefig(OUT / ('plots/shap_features_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()names = list(DATASETS.keys())if len(names) >= 2:    print('\nFeature overlap:')    for i in range(len(names)):        for j in range(i + 1, len(names)):            s1 = set(DATASETS[names[i]]['features'])            s2 = set(DATASETS[names[j]]['features'])            ov = s1 & s2            jc = len(ov) / len(s1 | s2) if (s1 | s2) else 0            print(' ', names[i], 'n', names[j], ':', len(ov), 'Jaccard=', round(jc, 3))            print('  Shared:', sorted(ov))

In [ ]:
def prepare_dataset(ds, name):    X     = ds['X'][ds['features']].values    y     = ds['y']    le    = ds['le']    n_cls = len(le.classes_)    ds['class_names'] = list(le.classes_)    print(' ', name, 'classes:', n_cls, list(le.classes_))    scaler = RobustScaler()    X_sc   = scaler.fit_transform(X)    X_tr, X_te, y_tr, y_te = train_test_split(        X_sc, y, test_size=TEST_SIZE,        random_state=RANDOM_STATE, stratify=y    )    print('  train:', len(X_tr), 'test:', len(X_te))    if n_cls >= 2:        cc     = Counter(y_tr)        target = max(500, int(np.median(list(cc.values()))))        strat  = {c: max(v, min(target, 3000)) for c, v in cc.items() if v < target}        if strat:            try:                sm = SMOTE(sampling_strategy=strat, random_state=RANDOM_STATE)                X_tr, y_tr = sm.fit_resample(X_tr, y_tr)                print('  SMOTE applied, train:', len(X_tr))            except Exception as e:                print('  SMOTE skipped:', e)    joblib.dump(scaler, OUT / ('models/scaler_' + name + '.joblib'))    return X_tr, X_te, y_tr, y_te, scalerfor name, ds in DATASETS.items():    print('\nPreparing', name)    X_tr, X_te, y_tr, y_te, scaler = prepare_dataset(ds, name)    ds.update({'X_train': X_tr, 'X_test': X_te,               'y_train': y_tr, 'y_test': y_te, 'scaler': scaler})print('\nAll datasets prepared')

In [ ]:
try:    import torch    use_gpu     = torch.cuda.is_available()    device      = 'cuda' if use_gpu else 'cpu'    tree_method = 'hist'    print('GPU:', use_gpu, 'tree_method:', tree_method, 'device:', device)except Exception:    use_gpu = False; device = 'cpu'; tree_method = 'hist'ALL_RESULTS = {}for name, ds in DATASETS.items():    print('\n' + '='*55)    print('Training on', name)    print('='*55)    X_tr, X_te = ds['X_train'], ds['X_test']    y_tr, y_te = ds['y_train'], ds['y_test']    le         = ds['le']    n_cls      = len(le.classes_)    if len(np.unique(y_tr)) < 2:        print('  SKIP — fewer than 2 classes')        continue    objective = 'binary:logistic' if n_cls == 2 else 'multi:softprob'    metric    = 'logloss'          if n_cls == 2 else 'mlogloss'    print(' ', n_cls, 'classes | objective:', objective)    results = {}    print('  XGBoost...')    xgb_m = xgb.XGBClassifier(        n_estimators=XGB_EPOCHS, max_depth=8, learning_rate=0.1,        subsample=0.8, colsample_bytree=0.8,        reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3,        tree_method=tree_method, device=device, n_jobs=-1,        random_state=RANDOM_STATE, objective=objective,        eval_metric=metric, early_stopping_rounds=PATIENCE, verbosity=0,    )    xgb_m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)    p = xgb_m.predict(X_te)    results['XGBoost'] = {'model': xgb_m, 'pred': p,        'acc': accuracy_score(y_te, p),        'f1':  f1_score(y_te, p, average='weighted', zero_division=0),        'precision': precision_score(y_te, p, average='weighted', zero_division=0),        'recall':    recall_score(y_te, p, average='weighted', zero_division=0)}    print('  Acc={:.4f} F1={:.4f}'.format(results['XGBoost']['acc'], results['XGBoost']['f1']))    joblib.dump(xgb_m, OUT / ('models/xgb_' + name + '.joblib'))    print('  Random Forest...')    rf_m = RandomForestClassifier(        n_estimators=200, max_depth=20, min_samples_split=5,        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)    rf_m.fit(X_tr, y_tr)    p = rf_m.predict(X_te)    results['RandomForest'] = {'model': rf_m, 'pred': p,        'acc': accuracy_score(y_te, p),        'f1':  f1_score(y_te, p, average='weighted', zero_division=0),        'precision': precision_score(y_te, p, average='weighted', zero_division=0),        'recall':    recall_score(y_te, p, average='weighted', zero_division=0)}    print('  Acc={:.4f} F1={:.4f}'.format(results['RandomForest']['acc'], results['RandomForest']['f1']))    joblib.dump(rf_m, OUT / ('models/rf_' + name + '.joblib'))    print('  LightGBM...')    lgb_m = lgb.LGBMClassifier(        n_estimators=300, max_depth=8, learning_rate=0.1,        subsample=0.8, colsample_bytree=0.8,        class_weight='balanced', n_jobs=-1,        random_state=RANDOM_STATE, verbosity=-1)    lgb_m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],              callbacks=[lgb.early_stopping(20, verbose=False)])    p = lgb_m.predict(X_te)    results['LightGBM'] = {'model': lgb_m, 'pred': p,        'acc': accuracy_score(y_te, p),        'f1':  f1_score(y_te, p, average='weighted', zero_division=0),        'precision': precision_score(y_te, p, average='weighted', zero_division=0),        'recall':    recall_score(y_te, p, average='weighted', zero_division=0)}    print('  Acc={:.4f} F1={:.4f}'.format(results['LightGBM']['acc'], results['LightGBM']['f1']))    joblib.dump(lgb_m, OUT / ('models/lgb_' + name + '.joblib'))    print('  Ensemble...')    xgb_ens = xgb.XGBClassifier(        n_estimators=xgb_m.best_iteration + 1,        max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,        reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3,        tree_method=tree_method, device=device, n_jobs=-1,        random_state=RANDOM_STATE, objective=objective,        eval_metric=metric, verbosity=0)    ens = VotingClassifier(        estimators=[('xgb', xgb_ens), ('rf', rf_m), ('lgb', lgb_m)],        voting='soft', n_jobs=-1)    ens.fit(X_tr, y_tr)    p = ens.predict(X_te)    results['Ensemble'] = {'model': ens, 'pred': p,        'acc': accuracy_score(y_te, p),        'f1':  f1_score(y_te, p, average='weighted', zero_division=0),        'precision': precision_score(y_te, p, average='weighted', zero_division=0),        'recall':    recall_score(y_te, p, average='weighted', zero_division=0)}    print('  Acc={:.4f} F1={:.4f}'.format(results['Ensemble']['acc'], results['Ensemble']['f1']))    joblib.dump(ens, OUT / ('models/ensemble_' + name + '.joblib'))    ALL_RESULTS[name]           = results    DATASETS[name]['best_model'] = xgb_mprint('\nAll models trained')

In [ ]:
comparison_rows = []for name, ds in DATASETS.items():    if name not in ALL_RESULTS:        continue    print('\n' + '='*55)    print('Evaluation:', name)    print('='*55)    le          = ds['le']    class_names = ds['class_names']    y_te        = ds['y_test']    res         = ALL_RESULTS[name]    print('{:<18} {:>10} {:>8} {:>10} {:>8}'.format('Model','Accuracy','F1','Precision','Recall'))    print('-' * 56)    for mname, r in res.items():        if mname in ('model', 'pred'):            continue        print('{:<18} {:>10.4f} {:>8.4f} {:>10.4f} {:>8.4f}'.format(            mname, r['acc'], r['f1'], r['precision'], r['recall']))        comparison_rows.append({'dataset': name, 'model': mname,            'accuracy': r['acc'], 'f1': r['f1'],            'precision': r['precision'], 'recall': r['recall']})    best_pred = res['XGBoost']['pred']    print('\nClassification Report (XGBoost):')    print(classification_report(y_te, best_pred, target_names=class_names, zero_division=0))    cm  = confusion_matrix(y_te, best_pred)    dim = max(8, len(class_names))    fig, ax = plt.subplots(figsize=(dim, dim - 1))    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(        ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)    ax.set_title('Confusion Matrix - ' + name + ' (XGBoost)')    plt.tight_layout()    plt.savefig(OUT / ('plots/confusion_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()    FP  = cm.sum(axis=0) - np.diag(cm)    TN  = cm.sum() - (FP + (cm.sum(axis=1) - np.diag(cm)) + np.diag(cm))    FPR = FP / (FP + TN + 1e-9)    fpr_df = pd.DataFrame({'class': class_names, 'FPR': FPR})    fpr_df.to_csv(OUT / ('plots/fpr_' + name + '.csv'), index=False)    print('Per-class FPR:')    for _, row in fpr_df.iterrows():        print('  {:<35} FPR={:.5f}'.format(str(row['class']), row['FPR']))comp_df = pd.DataFrame(comparison_rows)comp_df.to_csv(OUT / 'plots/model_comparison_all_datasets.csv', index=False)if len(DATASETS) > 1 and not comp_df.empty:    ens_df  = comp_df[comp_df['model'] == 'Ensemble'].copy()    min_val = min(ens_df['accuracy'].min(), ens_df['f1'].min())    y_lo    = max(0.0, min_val - 0.05)    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    ens_df.plot.bar(x='dataset', y='accuracy', ax=axes[0], color='#3B7DC8', legend=False)    axes[0].set_title('Ensemble Accuracy per Dataset')    axes[0].set_ylim(y_lo, 1.0)    ens_df.plot.bar(x='dataset', y='f1', ax=axes[1], color='#2ECC71', legend=False)    axes[1].set_title('Ensemble F1 per Dataset')    axes[1].set_ylim(y_lo, 1.0)    plt.tight_layout()    plt.savefig(OUT / 'plots/cross_dataset_comparison.png', dpi=150, bbox_inches='tight')    plt.show()print('\nEvaluation complete')

In [ ]:
SHAP_DATA = {}for name, ds in DATASETS.items():    if name not in ALL_RESULTS:        continue    print('\nSHAP analysis:', name)    model       = ds['best_model']    X_te        = ds['X_test']    y_te        = ds['y_test']    features    = ds['features']    rng = np.random.RandomState(RANDOM_STATE)    n_s = min(2000, len(X_te))    idx = rng.choice(len(X_te), n_s, replace=False)    Xs  = X_te[idx]    ys  = y_te[idx]    explainer   = shap.TreeExplainer(model)    shap_values = explainer.shap_values(Xs)    if isinstance(shap_values, list):        mean_abs = np.mean([np.abs(s).mean(0) for s in shap_values], axis=0)    elif shap_values.ndim == 3:        mean_abs = np.abs(shap_values).mean(axis=(0, 2))    else:        mean_abs = np.abs(shap_values).mean(0)    shap_imp = pd.Series(mean_abs, index=features).sort_values(ascending=False)    plt.figure(figsize=(9, 6))    shap_imp.head(15).sort_values().plot(kind='barh', color='#3B7DC8')    plt.title(name + ' — SHAP Global Feature Importance')    plt.xlabel('Mean |SHAP value| (all classes)')    plt.tight_layout()    plt.savefig(OUT / ('plots/shap_global_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()    sv_plot = shap_values    if isinstance(shap_values, list):        sv_plot = shap_values[0]    elif shap_values.ndim == 3:        sv_plot = shap_values[:, :, 0]    plt.figure()    shap.summary_plot(sv_plot, Xs, feature_names=features,                      plot_type='dot', show=False, max_display=12)    plt.title(name + ' — SHAP Beeswarm')    plt.tight_layout()    plt.savefig(OUT / ('plots/shap_beeswarm_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()    shap_imp.to_csv(OUT / ('explanations/shap_importance_' + name + '.csv'))    SHAP_DATA[name] = {'explainer': explainer, 'shap_values': shap_values,                       'X_shap': Xs, 'y_shap': ys,                       'shap_imp': shap_imp, 'features': features}    print('  SHAP done:', name)dsn = list(SHAP_DATA.keys())if len(dsn) >= 2:    print('\nCross-dataset SHAP top-10 overlap:')    for i in range(len(dsn)):        for j in range(i + 1, len(dsn)):            t1 = set(SHAP_DATA[dsn[i]]['shap_imp'].head(10).index)            t2 = set(SHAP_DATA[dsn[j]]['shap_imp'].head(10).index)            ov = t1 & t2            jc = len(ov) / len(t1 | t2) if (t1 | t2) else 0            print(' ', dsn[i], 'vs', dsn[j], ': overlap', len(ov), 'Jaccard=', round(jc, 3))            print('  Shared:', sorted(ov))

In [ ]:
LOCAL_SHAP = {}for name, sd in SHAP_DATA.items():    print('\nSHAP local waterfall:', name)    ds          = DATASETS[name]    class_names = ds['class_names']    shap_values = sd['shap_values']    Xs          = sd['X_shap']    ys          = sd['y_shap']    features    = sd['features']    explainer   = sd['explainer']    model       = ds['best_model']    LOCAL_SHAP[name] = {}    for cls_id in np.unique(ys)[:5]:        cls_name = class_names[cls_id]        cidx     = np.where(ys == cls_id)[0]        if len(cidx) == 0:            continue        si = cidx[0]        if isinstance(shap_values, list):            sv_local = shap_values[cls_id][si]            bv = (explainer.expected_value[cls_id]                  if hasattr(explainer.expected_value, '__len__')                  else explainer.expected_value)        elif shap_values.ndim == 3:            sv_local = shap_values[si, :, cls_id]            bv = (explainer.expected_value[cls_id]                  if hasattr(explainer.expected_value, '__len__')                  else explainer.expected_value)        else:            sv_local = shap_values[si]            bv = explainer.expected_value        exp_obj = shap.Explanation(            values=sv_local, base_values=float(bv),            data=Xs[si], feature_names=features)        pred_id   = int(model.predict(Xs[si:si + 1])[0])        pred_name = class_names[pred_id]        tick      = 'OK' if pred_id == cls_id else 'WRONG'        plt.figure()        shap.waterfall_plot(exp_obj, max_display=10, show=False)        safe = cls_name.replace(' ', '_').replace('/', '-')        plt.title(name + ' | ' + cls_name + ' -> ' + pred_name + ' ' + tick)        plt.tight_layout()        plt.savefig(OUT / ('plots/shap_waterfall_' + name + '_' + safe + '.png'),                    dpi=150, bbox_inches='tight')        plt.show()        LOCAL_SHAP[name][cls_name] = {            'shap_local': sv_local,            'shap_top5':  list(pd.Series(np.abs(sv_local),                               index=features).nlargest(5).index)}    print('  local SHAP done:', name)

In [ ]:
LIME_DATA = {}for name, ds in DATASETS.items():    if name not in SHAP_DATA:        continue    print('\nLIME explanations:', name)    le          = ds['le']    class_names = ds['class_names']    model       = ds['best_model']    features    = ds['features']    sd          = SHAP_DATA[name]    Xs          = sd['X_shap']    ys          = sd['y_shap']    lime_exp = lime.lime_tabular.LimeTabularExplainer(        training_data=ds['X_train'][:5000],        feature_names=features, class_names=class_names,        mode='classification', discretize_continuous=True,        random_state=RANDOM_STATE)    LIME_DATA[name] = {}    for cls_id in np.unique(ys)[:3]:        cls_name = class_names[cls_id]        cidx     = np.where(ys == cls_id)[0]        if len(cidx) == 0:            continue        si = cidx[0]        print('  Explaining:', cls_name)        try:            exp     = lime_exp.explain_instance(                Xs[si], model.predict_proba,                num_features=10, labels=(cls_id,), num_samples=500)            weights = dict(exp.as_list(label=cls_id))            def extract_feat(cond, feat_list):                for f in feat_list:                    if f in cond:                        return f                return cond            lime_top5 = list(                pd.Series({k: abs(v) for k, v in weights.items()})                .nlargest(5).index)            lime_top5_clean = [extract_feat(f, features) for f in lime_top5]            safe = cls_name.replace(' ', '_').replace('/', '-')            exp.save_to_file(str(OUT / ('explanations/lime_' + name + '_' + safe + '.html')))            feat_l = list(weights.keys())[:8]            vals_l = [weights[f] for f in feat_l]            cols   = ['#E74C3C' if v < 0 else '#2ECC71' for v in vals_l]            plt.figure(figsize=(9, 4))            plt.barh(feat_l, vals_l, color=cols)            plt.axvline(0, color='gray', linewidth=0.8)            plt.title(name + ' | LIME - ' + cls_name)            plt.xlabel('Feature weight')            plt.tight_layout()            plt.savefig(OUT / ('plots/lime_' + name + '_' + safe + '.png'),                        dpi=150, bbox_inches='tight')            plt.show()            LIME_DATA[name][cls_name] = {'weights': weights,                'lime_top5': lime_top5, 'lime_top5_clean': lime_top5_clean}        except Exception as e:            print('  LIME failed for', cls_name, ':', e)    print('  LIME done:', name)

In [ ]:
JACCARD_ALL = []for name in DATASETS:    if name not in SHAP_DATA or name not in LIME_DATA:        continue    print('  Dataset:', name)    for cls_name in LOCAL_SHAP.get(name, {}):        if cls_name not in LIME_DATA[name]:            continue        shap5 = set(LOCAL_SHAP[name][cls_name]['shap_top5'])        lime5 = set(LIME_DATA[name][cls_name]['lime_top5_clean'])        inter = shap5 & lime5        union = shap5 | lime5        jac   = len(inter) / len(union) if union else 0        JACCARD_ALL.append({'dataset': name, 'class': cls_name,            'shap_top5': list(shap5), 'lime_top5': list(lime5),            'overlap': list(inter), 'jaccard': jac})        print('   {:<30} Jaccard={:.3f}  Overlap={}'.format(cls_name, jac, list(inter)))jac_df = pd.DataFrame(JACCARD_ALL)if not jac_df.empty:    jac_df.to_csv(OUT / 'explanations/shap_lime_jaccard_all.csv', index=False)    print('\nMean Jaccard per dataset:')    for n in jac_df['dataset'].unique():        mj = jac_df[jac_df['dataset'] == n]['jaccard'].mean()        print('  ', n, ':', round(mj, 3))    print('  Overall:', round(jac_df['jaccard'].mean(), 3))    plt.figure(figsize=(10, 5))    for dn in jac_df['dataset'].unique():        sub = jac_df[jac_df['dataset'] == dn]        plt.bar(sub['class'], sub['jaccard'], label=dn, alpha=0.7)    plt.axhline(0.5, color='red', linestyle='--', linewidth=0.8)    plt.title('SHAP vs LIME Jaccard Similarity (top-5 features)')    plt.ylabel('Jaccard Similarity')    plt.xticks(rotation=45, ha='right')    plt.legend()    plt.tight_layout()    plt.savefig(OUT / 'plots/jaccard_shap_lime.png', dpi=150, bbox_inches='tight')    plt.show()print('\nCross-validation complete')

In [ ]:
def compute_xcs(model, explainer, lime_explainer,                X_sample, y_sample, le, features,                k=5, n_neighbours=5,                w1=XCS_W1, w2=XCS_W2, w3=XCS_W3):    records   = []    proba_all = model.predict_proba(X_sample)    preds_all = model.predict(X_sample)    sv_all    = explainer.shap_values(X_sample)    for i in range(len(X_sample)):        pred_id = int(preds_all[i])        true_id = int(y_sample[i])        conf    = float(proba_all[i, pred_id])        if isinstance(sv_all, list):            sv_i = sv_all[pred_id][i]        elif sv_all.ndim == 3:            sv_i = sv_all[i, :, pred_id]        else:            sv_i = sv_all[i]        mean_abs   = np.mean(np.abs(sv_i)) + 1e-9        shap_top_k = set(pd.Series(np.abs(sv_i), index=features).nlargest(k).index)        nbr_svs = []        for _ in range(n_neighbours):            x_n  = X_sample[i] + np.random.normal(0, 0.05, X_sample[i].shape)            sv_n = explainer.shap_values(x_n.reshape(1, -1))            if isinstance(sv_n, list):                sv_ni = sv_n[pred_id][0]            elif sv_n.ndim == 3:                sv_ni = sv_n[0, :, pred_id]            else:                sv_ni = sv_n[0]            nbr_svs.append(np.abs(sv_ni))        instab_norm = min(float(np.std(nbr_svs)) / mean_abs, 0.5)        try:            lime_e = lime_explainer.explain_instance(                X_sample[i], model.predict_proba,                num_features=k, labels=(pred_id,), num_samples=200)            lw = dict(lime_e.as_list(label=pred_id))            lime_top_k = set(                pd.Series({kk: abs(v) for kk, v in lw.items()}).nlargest(k).index)        except Exception:            lime_top_k = set()        union      = shap_top_k | lime_top_k        jaccard_sl = len(shap_top_k & lime_top_k) / len(union) if union else 0.0        xcs = w1 * conf + w2 * (1.0 - instab_norm) + w3 * jaccard_sl        true_name = le.classes_[true_id] if true_id < len(le.classes_) else str(true_id)        records.append({'sample_id': i,            'true_class': true_name, 'pred_class': le.classes_[pred_id],            'correct': pred_id == true_id, 'confidence': round(conf, 4),            'shap_instability': round(instab_norm, 4), 'jaccard_sl': round(jaccard_sl, 4),            'xcs': round(xcs, 4), 'flag_review': bool(xcs < XCS_THRESHOLD)})    return pd.DataFrame(records)XCS_RESULTS = {}for name, ds in DATASETS.items():    if name not in SHAP_DATA or name not in LIME_DATA:        print('Skipping XCS for', name)        continue    print('\nComputing XCS:', name, '(', N_XCS_SAMPLES, 'samples)')    le       = ds['le']    model    = ds['best_model']    features = ds['features']    X_te     = ds['X_test']    y_te     = ds['y_test']    explainer= SHAP_DATA[name]['explainer']    lime_exp = lime.lime_tabular.LimeTabularExplainer(        training_data=ds['X_train'][:3000],        feature_names=features, class_names=ds['class_names'],        mode='classification', discretize_continuous=True,        random_state=RANDOM_STATE)    unique_cls = np.unique(y_te)    n_per      = max(1, N_XCS_SAMPLES // len(unique_cls))    idx        = []    for cls in unique_cls:        cidx = np.where(y_te == cls)[0]        idx.extend(cidx[:n_per].tolist())    idx = idx[:N_XCS_SAMPLES]    xcs_df = compute_xcs(model, explainer, lime_exp,                         X_te[idx], y_te[idx], le, features,                         k=5, n_neighbours=5)    XCS_RESULTS[name] = xcs_df    xcs_df.to_csv(OUT / ('explanations/xcs_' + name + '.csv'), index=False)    mean_xcs  = xcs_df['xcs'].mean()    n_flagged = xcs_df['flag_review'].sum()    corr_xcs  = xcs_df[xcs_df['correct']]['xcs'].mean()    wrong_xcs = xcs_df[~xcs_df['correct']]['xcs'].mean()    print('  Mean XCS:', round(mean_xcs, 3))    print('  Flagged:', n_flagged, '/', len(xcs_df), '(threshold =', XCS_THRESHOLD, ')')    print('  XCS correct:', round(corr_xcs, 3))    print('  XCS wrong  :', round(wrong_xcs, 3) if not np.isnan(wrong_xcs) else 'N/A')    fig, axes = plt.subplots(1, 2, figsize=(13, 4))    axes[0].hist(xcs_df[xcs_df['correct']]['xcs'],                 bins=20, alpha=0.7, color='#2ECC71', label='Correct')    axes[0].hist(xcs_df[~xcs_df['correct']]['xcs'],                 bins=20, alpha=0.7, color='#E74C3C', label='Wrong')    axes[0].axvline(XCS_THRESHOLD, color='gray', linestyle='--',                    linewidth=1, label='Threshold ' + str(XCS_THRESHOLD))    axes[0].set_title(name + ' — XCS Distribution')    axes[0].set_xlabel('XCS score')    axes[0].legend(fontsize=9)    xcs_df.groupby('pred_class')['xcs'].mean().sort_values().plot(        kind='barh', ax=axes[1], color='#3B7DC8')    axes[1].axvline(XCS_THRESHOLD, color='gray', linestyle='--', linewidth=1)    axes[1].set_title(name + ' — Mean XCS per Attack Class')    axes[1].set_xlabel('Mean XCS')    plt.tight_layout()    plt.savefig(OUT / ('plots/xcs_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()    plt.figure(figsize=(7, 5))    colors = xcs_df['correct'].map({True: '#2ECC71', False: '#E74C3C'})    plt.scatter(xcs_df['confidence'], xcs_df['xcs'], c=colors, alpha=0.6, s=40)    plt.axhline(XCS_THRESHOLD, color='gray', linestyle='--', linewidth=0.8)    plt.xlabel('Model Confidence')    plt.ylabel('XCS Score')    plt.title(name + ' — XCS vs Confidence')    plt.legend(handles=[        mpatches.Patch(color='#2ECC71', label='Correct'),        mpatches.Patch(color='#E74C3C', label='Wrong')])    plt.tight_layout()    plt.savefig(OUT / ('plots/xcs_scatter_' + name + '.png'), dpi=150, bbox_inches='tight')    plt.show()print('\nXCS complete')

In [ ]:
import jsonclass NpEncoder(json.JSONEncoder):    def default(self, o):        if isinstance(o, np.integer): return int(o)        if isinstance(o, np.floating): return float(o)        if isinstance(o, np.ndarray): return o.tolist()        return super().default(o)jac_mean = float(jac_df['jaccard'].mean()) if (len(JACCARD_ALL) > 0) else Nonemetadata = {    'project': 'XAI-IDS: Explainable AI Intrusion Detection',    'repo':    'MohammadThabetHassan/xai-ids-cicids2017',    'xai_methods': ['SHAP TreeExplainer', 'LIME TabularExplainer'],    'models': ['XGBoost', 'RandomForest', 'LightGBM', 'VotingEnsemble'],    'xcs': {'formula': 'XCS = w1*Conf + w2*(1-Instab) + w3*Jaccard_SL',            'weights': {'w1': XCS_W1, 'w2': XCS_W2, 'w3': XCS_W3},            'threshold': XCS_THRESHOLD},    'cross_dataset_jaccard': jac_mean,    'datasets': {},}for name, ds in DATASETS.items():    if name not in ALL_RESULTS:        continue    res = ALL_RESULTS[name]    metadata['datasets'][name] = {        'n_features': len(ds['features']),        'features':   ds['features'],        'n_classes':  len(ds['le'].classes_),        'classes':    list(ds['le'].classes_),        'results': {            m: {'accuracy': float(r['acc']), 'f1': float(r['f1']),                'precision': float(r['precision']), 'recall': float(r['recall'])}            for m, r in res.items() if m not in ('model', 'pred')        }    }    joblib.dump(ds['le'],       OUT / ('models/label_encoder_' + name + '.joblib'))    joblib.dump(ds['features'], OUT / ('models/features_' + name + '.joblib'))with open(OUT / 'model_metadata.json', 'w') as f:    json.dump(metadata, f, indent=2, cls=NpEncoder)print('Output files:')for f in sorted(OUT.rglob('*')):    if f.is_file():        sz   = f.stat().st_size        unit = str(round(sz / 1e6, 1)) + 'MB' if sz > 1e6 else str(round(sz / 1e3)) + 'KB'        print(' ', str(f.relative_to(OUT)), '(', unit, ')')print('\nAll artifacts saved')

In [ ]:
import shutilprimary_ds = max(    [n for n in ALL_RESULTS],    key=lambda n: ALL_RESULTS[n]['Ensemble']['f1'])print('FastAPI primary dataset:', primary_ds)main_code = (    'import joblib, json, numpy as np\n'    'from fastapi import FastAPI, HTTPException\n'    'from pydantic import BaseModel\n'    'from typing import List, Optional\n'    'import shap\n\n'    'app = FastAPI(title="XAI-IDS", version="2.0.0")\n\n'    'model    = joblib.load("model/xgb_' + primary_ds + '.joblib")\n'    'scaler   = joblib.load("model/scaler_' + primary_ds + '.joblib")\n'    'le       = joblib.load("model/label_encoder_' + primary_ds + '.joblib")\n'    'features = joblib.load("model/features_' + primary_ds + '.joblib")\n'    'explainer = shap.TreeExplainer(model)\n'    'with open("model/model_metadata.json") as f: meta = json.load(f)\n\n'    'class Input(BaseModel):\n'    '    features: List[float]\n'    '    explain: bool = True\n\n'    'class Output(BaseModel):\n'    '    prediction: str\n'    '    confidence: float\n'    '    is_attack: bool\n'    '    shap_top10: Optional[dict] = None\n\n'    '@app.get("/")\ndef root(): return meta\n\n'    '@app.get("/health")\ndef health(): return {"status": "ok"}\n\n'    '@app.post("/predict", response_model=Output)\n'    'def predict(data: Input):\n'    '    if len(data.features) != len(features):\n'    '        raise HTTPException(422, f"Expected {len(features)} features")\n'    '    X = np.array(data.features).reshape(1,-1)\n'    '    Xs = scaler.transform(X)\n'    '    pid = model.predict(Xs)[0]\n'    '    prob = model.predict_proba(Xs)[0]\n'    '    cls = le.classes_[pid]\n'    '    shd = None\n'    '    if data.explain:\n'    '        sv = explainer.shap_values(Xs)\n'    '        arr = sv[pid][0] if isinstance(sv,list) else (sv[0,:,pid] if sv.ndim==3 else sv[0])\n'    '        top = np.argsort(np.abs(arr))[::-1][:10]\n'    '        shd = {features[i]: float(arr[i]) for i in top}\n'    '    return Output(prediction=cls, confidence=float(prob[pid]),\n'    '                  is_attack=cls.lower() not in ["benign","normal"],\n'    '                  shap_top10=shd)\n')dockerfile = (    'FROM python:3.10-slim\nWORKDIR /app\n'    'COPY requirements.txt .\nRUN pip install -r requirements.txt\n'    'COPY . .\nEXPOSE 8000\n'    'CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]\n')reqs = (    'fastapi==0.111.0\nuvicorn==0.30.0\npydantic==2.7.0\n'    'joblib==1.4.2\nnumpy==1.26.4\nscikit-learn==1.5.0\n'    'xgboost==2.0.3\nshap==0.45.0\n')api_dir = OUT / 'fastapi_app'(api_dir / 'model').mkdir(parents=True, exist_ok=True)(api_dir / 'main.py').write_text(main_code)(api_dir / 'Dockerfile').write_text(dockerfile)(api_dir / 'requirements.txt').write_text(reqs)for f in sorted((OUT / 'models').iterdir()):    shutil.copy2(f, api_dir / 'model' / f.name)shutil.copy2(OUT / 'model_metadata.json', api_dir / 'model' / 'model_metadata.json')print('FastAPI app written to', api_dir)print('Run: uvicorn main:app --reload')

In [ ]:
sep = '=' * 70print(sep)print('  XAI-IDS -- RESEARCH PAPER RESULTS SUMMARY')print(sep)print('  XAI Methods:', 'SHAP (TreeExplainer) + LIME (TabularExplainer)')print('  Models     :', 'XGBoost + RandomForest + LightGBM + VotingEnsemble')print('  Datasets   :', list(DATASETS.keys()))print()print('{:<18} {:<16} {:>10} {:>8} {:>8} {:>8}'.format(    'Dataset', 'Model', 'Accuracy', 'F1', 'P', 'R'))print('-' * 70)for name, res in ALL_RESULTS.items():    for mname, r in res.items():        if mname in ('model', 'pred'):            continue        print('{:<18} {:<16} {:>10.4f} {:>8.4f} {:>8.4f} {:>8.4f}'.format(            name, mname, r['acc'], r['f1'], r['precision'], r['recall']))    print()if len(JACCARD_ALL) > 0:    print('SHAP vs LIME Jaccard:')    for n in jac_df['dataset'].unique():        mj = jac_df[jac_df['dataset'] == n]['jaccard'].mean()        print(' ', n, ':', round(mj, 3))    print('  Overall:', round(jac_df['jaccard'].mean(), 3))    print()if XCS_RESULTS:    print('XCS Summary (threshold =', XCS_THRESHOLD, '):')    for n, df in XCS_RESULTS.items():        c = df[df['correct']]['xcs'].mean()        w = df[~df['correct']]['xcs'].mean()        print(' ', n, '| mean=', round(df['xcs'].mean(), 3),              '| correct=', round(c, 3),              '| wrong=', round(w, 3) if not np.isnan(w) else 'N/A',              '| flagged=', df['flag_review'].sum(), '/', len(df))    print()print('Research contributions:')print('  1. Multi-dataset cross-dataset generalization (3 datasets)')print('  2. Novel XCS metric: confidence + stability + SHAP-LIME agreement')print('  3. SHAP + LIME dual XAI with Jaccard cross-validation')print('  4. Per-class FPR analysis across all attack types')print('  5. ANOVA + SHAP two-stage feature selection')print('  6. Voting ensemble (XGBoost + RF + LightGBM)')print('  7. FastAPI deployment-ready artifacts')print(sep)import subprocessimport shutilimport refrom pathlib import PathREPO_DIR     = "/tmp/xai-ids-repo"GITHUB_TOKEN = "YOUR_TOKEN_HERE"   # ← paste your token hereYOUR_EMAIL   = "20220002188@students.cud.ac.ae"YOUR_NAME    = "MohammadThabetHassan"OUT          = Path('/kaggle/working/xai_ids_outputs')REPO         = Path(REPO_DIR)# ── Clone fresh ──────────────────────────────────subprocess.run(['rm', '-rf', REPO_DIR])new_url = f"https://{GITHUB_TOKEN}@github.com/MohammadThabetHassan/xai-ids-cicids2017.git"subprocess.run(['git', 'clone', new_url, REPO_DIR], check=True)subprocess.run(['git', 'config', 'user.email', YOUR_EMAIL], cwd=REPO_DIR, check=True)subprocess.run(['git', 'config', 'user.name',  YOUR_NAME],  cwd=REPO_DIR, check=True)# ── Copy notebook (scrub token first) ────────────nb_src = '/kaggle/working/.virtual_documents/__notebook_source__.ipynb'nb_text = open(nb_src).read()nb_text = re.sub(r'ghp_[A-Za-z0-9]{36}', 'YOUR_TOKEN_HERE', nb_text)nb_text = re.sub(r'(GITHUB_TOKEN\s*=\s*)["\'].*?["\']', r'\1"YOUR_TOKEN_HERE"', nb_text)(REPO / 'xai_ids_multidataset.ipynb').write_text(nb_text)print('✅ Notebook copied (token scrubbed)')# ── Copy metadata ─────────────────────────────────shutil.copy(str(OUT / 'model_metadata.json'), str(REPO / 'model_metadata.json'))# ── Copy plots ────────────────────────────────────plots_dst = REPO / 'plots'plots_dst.mkdir(exist_ok=True)for f in (OUT / 'plots').glob('*'):    shutil.copy(str(f), str(plots_dst / f.name))print(f'✅ Copied {len(list((OUT/"plots").glob("*")))} plot files')# ── Copy explanations ─────────────────────────────exp_dst = REPO / 'explanations'exp_dst.mkdir(exist_ok=True)for f in (OUT / 'explanations').glob('*'):    shutil.copy(str(f), str(exp_dst / f.name))print(f'✅ Copied {len(list((OUT/"explanations").glob("*")))} explanation files')# ── Copy small models only (<50MB) ───────────────models_dst = REPO / 'models'models_dst.mkdir(exist_ok=True)skipped = []for f in (OUT / 'models').glob('*'):    if f.stat().st_size / 1e6 < 50:        shutil.copy(str(f), str(models_dst / f.name))    else:        skipped.append(f'{f.name} ({f.stat().st_size/1e6:.0f}MB)')print(f'✅ Models copied. Skipped large: {skipped}')# ── Update .gitignore ─────────────────────────────(REPO / '.gitignore').write_text(    "# Large model files — too big for GitHub\n"    "models/ensemble_*.joblib\n"    "models/rf_UNSWNB15.joblib\n"    "models/rf_CICIDS2017.joblib\n"    "fastapi_app/\n"    "__pycache__/\n"    "*.pyc\n")# ── Stage + commit ────────────────────────────────subprocess.run(['git', 'add', '.'], cwd=REPO_DIR, check=True)commit = subprocess.run([    'git', 'commit', '-m',    'Final XAI-IDS v2: fixed Jaccard, XCS, class names — CICIDS2017/UNSWNB15/CICIDS2018'], cwd=REPO_DIR, capture_output=True, text=True)print(commit.stdout)if commit.returncode != 0:    print('Nothing to commit or error:', commit.stderr)# ── Push ──────────────────────────────────────────push = subprocess.run(    ['git', 'push', 'origin', 'main', '--force'],    cwd=REPO_DIR, capture_output=True, text=True)print(push.stdout)print(push.stderr)print('✅ Pushed!' if push.returncode == 0 else '❌ Push failed')